In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 1. Load the prepared four-class dataset
ROOT = Path("/content/drive/MyDrive/RoofMaterialProject")
INPUT = ROOT / "features/roof_features_4class.csv"

df = pd.read_csv(INPUT)

# 2. Select ONLY the 44 handcrafted numeric features
feature_cols = [
    c for c in df.columns
    if c.startswith((
        "r_", "g_", "b_", "h_", "s_", "v_",
        "lbp_", "glcm_"
    ))
]

X = df[feature_cols].astype(np.float32)
y = df["roof_material"]

assert len(feature_cols) == 44, f"Expected 44 features, found {len(feature_cols)}"
assert not X.isna().any().any(), "Missing feature values found"
assert np.isfinite(X.to_numpy()).all(), "Infinite feature values found"

print("Dataset shape:", X.shape)
print("Classes:", sorted(y.unique()))

# 3. Stratified 80/20 train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Validation samples:", len(X_val))

# 4. SVM pipeline: scaler learns from training data only
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        C=1.0,
        class_weight="balanced",
        random_state=42
    ))
])

# 5. Train
print("\nTraining SVM...")
svm_model.fit(X_train, y_train)

# 6. Evaluate on validation data
y_pred = svm_model.predict(X_val)

print("\n========== SVM BASELINE RESULTS ==========")
print(f"Accuracy:          {accuracy_score(y_val, y_pred):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_val, y_pred):.4f}")
print(f"Macro-F1:          {f1_score(y_val, y_pred, average='macro'):.4f}")

labels = [
    "concrete_cement",
    "healthy_metal",
    "incomplete",
    "irregular_metal"
]

print("\nClassification report:")
print(classification_report(
    y_val, y_pred,
    labels=labels,
    zero_division=0,
    digits=4
))

print("Confusion matrix (rows = actual, columns = predicted):")
print(labels)
print(confusion_matrix(y_val, y_pred, labels=labels))

# 7. Save metrics and validation predictions
results = pd.DataFrame({
    "building_id": df.loc[X_val.index, "building_id"].values,
    "region": df.loc[X_val.index, "region"].values,
    "actual": y_val.values,
    "predicted": y_pred
})

OUTPUT = ROOT / "features/svm_baseline_validation_predictions.csv"
results.to_csv(OUTPUT, index=False)

print("\nSaved validation predictions to:", OUTPUT)


Dataset shape: (10333, 44)
Classes: ['concrete_cement', 'healthy_metal', 'incomplete', 'irregular_metal']

Training samples: 8266
Validation samples: 2067

Training SVM...

========== SVM BASELINE RESULTS ==========
Accuracy:          0.5960
Balanced accuracy: 0.5573
Macro-F1:          0.4716

Classification report:
                 precision    recall  f1-score   support

concrete_cement     0.1707    0.4949    0.2539        99
  healthy_metal     0.8247    0.7169    0.7670      1109
     incomplete     0.2708    0.5909    0.3714       132
irregular_metal     0.5871    0.4264    0.4940       727

       accuracy                         0.5960      2067
      macro avg     0.4633    0.5573    0.4716      2067
   weighted avg     0.6744    0.5960    0.6212      2067

Confusion matrix (rows = actual, columns = predicted):
['concrete_cement', 'healthy_metal', 'incomplete', 'irregular_metal']
[[ 49  14  18  18]
 [ 97 795  38 179]
 [ 24   9  78  21]
 [117 146 154 310]]

Saved validation pre

In [6]:
"""hypertuning"""

'hypertuning'

In [7]:

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 1. Define the same SVM pipeline
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        class_weight="balanced"
    ))
])

# 2. Hyperparameter grid
param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.01, 0.1]
}

# 3. Cross-validation on training data ONLY
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True,
    return_train_score=True
)

# 4. Train and tune SVM
print("Starting SVM hyperparameter tuning...")
grid_search.fit(X_train, y_train)

# 5. Best settings from cross-validation
print("\n========== BEST SVM SETTINGS ==========")
print("Best parameters:", grid_search.best_params_)
print(f"Best CV macro-F1: {grid_search.best_score_:.4f}")

# 6. Evaluate the selected SVM on the SAME held-out validation set
best_svm = grid_search.best_estimator_
y_pred_tuned = best_svm.predict(X_val)

print("\n========== TUNED SVM RESULTS ==========")
print(f"Accuracy:          {accuracy_score(y_val, y_pred_tuned):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_val, y_pred_tuned):.4f}")
print(f"Macro-F1:          {f1_score(y_val, y_pred_tuned, average='macro'):.4f}")

labels = [
    "concrete_cement",
    "healthy_metal",
    "incomplete",
    "irregular_metal"
]

print("\nClassification report:")
print(classification_report(
    y_val,
    y_pred_tuned,
    labels=labels,
    zero_division=0,
    digits=4
))

print("Confusion matrix (rows = actual, columns = predicted):")
print(labels)
print(confusion_matrix(y_val, y_pred_tuned, labels=labels))

# 7. Compare baseline and tuned results
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Balanced Accuracy", "Macro-F1"],
    "Baseline SVM": [
        0.5960,
        0.5573,
        0.4716
    ],
    "Tuned SVM": [
        accuracy_score(y_val, y_pred_tuned),
        balanced_accuracy_score(y_val, y_pred_tuned),
        f1_score(y_val, y_pred_tuned, average="macro")
    ]
})

comparison["Change"] = (
    comparison["Tuned SVM"] - comparison["Baseline SVM"]
)

print("\n========== BASELINE VS TUNED ==========")
print(comparison.round(4).to_string(index=False))

# 8. Save CV results and tuned validation predictions
ROOT = "/content/drive/MyDrive/RoofMaterialProject"

cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results.to_csv(
    f"{ROOT}/features/svm_hyperparameter_cv_results.csv",
    index=False
)

validation_results = pd.DataFrame({
    "building_id": df.loc[X_val.index, "building_id"].values,
    "region": df.loc[X_val.index, "region"].values,
    "actual": y_val.values,
    "predicted": y_pred_tuned
})

validation_results.to_csv(
    f"{ROOT}/features/svm_tuned_validation_predictions.csv",
    index=False
)

print("\nSaved CV results and tuned predictions to Drive.")


Starting SVM hyperparameter tuning...
Fitting 3 folds for each of 9 candidates, totalling 27 fits

========== BEST SVM SETTINGS ==========
Best parameters: {'svm__C': 10, 'svm__gamma': 'scale'}
Best CV macro-F1: 0.4736

========== TUNED SVM RESULTS ==========
Accuracy:          0.6280
Balanced accuracy: 0.5374
Macro-F1:          0.4919

Classification report:
                 precision    recall  f1-score   support

concrete_cement     0.2079    0.4242    0.2791        99
  healthy_metal     0.8079    0.7358    0.7702      1109
     incomplete     0.3054    0.4697    0.3701       132
irregular_metal     0.5798    0.5199    0.5482       727

       accuracy                         0.6280      2067
      macro avg     0.4753    0.5374    0.4919      2067
   weighted avg     0.6668    0.6280    0.6430      2067

Confusion matrix (rows = actual, columns = predicted):
['concrete_cement', 'healthy_metal', 'incomplete', 'irregular_metal']
[[ 42  16  15  26]
 [ 52 816  28 213]
 [ 24  11  62  3

In [8]:

import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

labels = [
    "concrete_cement",
    "healthy_metal",
    "incomplete",
    "irregular_metal"
]

# Baseline predictions saved before tuning.
# Reconstruct them only if the original baseline model still exists.
# Best option: save baseline predictions before overwriting anything.
# If svm_model is still your original baseline model, use:
y_pred_baseline = svm_model.predict(X_val)

# Tuned SVM predictions already generated by GridSearchCV
y_pred_tuned = best_svm.predict(X_val)

# Calculate per-class metrics
p_base, r_base, f_base, support_base = precision_recall_fscore_support(
    y_val, y_pred_baseline, labels=labels, zero_division=0
)

p_tuned, r_tuned, f_tuned, support_tuned = precision_recall_fscore_support(
    y_val, y_pred_tuned, labels=labels, zero_division=0
)

comparison = pd.DataFrame({
    "Roof class": labels,
    "Support": support_tuned,
    "Baseline precision": p_base,
    "Tuned precision": p_tuned,
    "Precision change": p_tuned - p_base,
    "Baseline recall": r_base,
    "Tuned recall": r_tuned,
    "Recall change": r_tuned - r_base,
    "Baseline F1": f_base,
    "Tuned F1": f_tuned,
    "F1 change": f_tuned - f_base
})

print(comparison.round(4).to_string(index=False))

print("\nMacro-F1 baseline:",
      round(np.mean(f_base), 4))
print("Macro-F1 tuned:",
      round(np.mean(f_tuned), 4))


     Roof class  Support  Baseline precision  Tuned precision  Precision change  Baseline recall  Tuned recall  Recall change  Baseline F1  Tuned F1  F1 change
concrete_cement       99              0.1707           0.2079            0.0372           0.4949        0.4242        -0.0707       0.2539    0.2791     0.0252
  healthy_metal     1109              0.8247           0.8079           -0.0168           0.7169        0.7358         0.0189       0.7670    0.7702     0.0032
     incomplete      132              0.2708           0.3054            0.0346           0.5909        0.4697        -0.1212       0.3714    0.3701    -0.0013
irregular_metal      727              0.5871           0.5798           -0.0074           0.4264        0.5199         0.0935       0.4940    0.5482     0.0542

Macro-F1 baseline: 0.4716
Macro-F1 tuned: 0.4919


In [4]:
print("best_svm" in globals())

False


In [9]:
import joblib

save_path = "/content/drive/MyDrive/RoofMaterialProject/features/svm_tuned_experiment1.joblib"

joblib.dump(grid_search.best_estimator_, save_path)

print("Best SVM saved successfully!")
print("Best parameters:", grid_search.best_params_)

Best SVM saved successfully!
Best parameters: {'svm__C': 10, 'svm__gamma': 'scale'}


In [10]:
import os

save_path = "/content/drive/MyDrive/RoofMaterialProject/features/svm_tuned_experiment1.joblib"

print("File exists:", os.path.exists(save_path))
print("File size:", round(os.path.getsize(save_path) / (1024 * 1024), 2), "MB")

File exists: True
File size: 2.36 MB


In [12]:
print(
    "X_train:", "X_train" in globals(),
    "\nX_val:", "X_val" in globals(),
    "\ny_train:", "y_train" in globals(),
    "\ny_val:", "y_val" in globals()
)

X_train: True 
X_val: True 
y_train: True 
y_val: True


In [11]:
"""exp2 with diff parameters of hypertuning like 25 cobinations"""

'exp2 with diff parameters of hypertuning like 25 cobinations'

In [17]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm_pipeline_2 = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", class_weight="balanced"))
])

param_grid_2 = {
    "svm__C": [0.01, 0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.001, 0.01, 0.1, 1]
}

cv_2 = StratifiedKFold(
    n_splits=3, shuffle=True, random_state=42
)

grid_search_2 = GridSearchCV(
    svm_pipeline_2,
    param_grid_2,
    scoring="f1_macro",
    cv=cv_2,
    n_jobs=2,
    verbose=2,
    refit=True,
    return_train_score=True
)

grid_search_2.fit(X_train, y_train)

print("Best parameters:", grid_search_2.best_params_)
print("Best CV Macro-F1:", grid_search_2.best_score_)

Fitting 3 folds for each of 25 candidates, totalling 75 fits
Best parameters: {'svm__C': 10, 'svm__gamma': 'scale'}
Best CV Macro-F1: 0.4736101065783081


In [18]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

best_svm_2 = grid_search_2.best_estimator_

y_pred_2 = best_svm_2.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred_2))
print("Balanced Accuracy:", balanced_accuracy_score(y_val, y_pred_2))
print("Macro-F1:", f1_score(y_val, y_pred_2, average="macro"))

print("\nClassification Report:")
print(classification_report(y_val, y_pred_2))

Accuracy: 0.6279632317368167
Balanced Accuracy: 0.5374205973843739
Macro-F1: 0.49190424552310474

Classification Report:
                 precision    recall  f1-score   support

concrete_cement       0.21      0.42      0.28        99
  healthy_metal       0.81      0.74      0.77      1109
     incomplete       0.31      0.47      0.37       132
irregular_metal       0.58      0.52      0.55       727

       accuracy                           0.63      2067
      macro avg       0.48      0.54      0.49      2067
   weighted avg       0.67      0.63      0.64      2067



In [19]:
import joblib
import os
import pandas as pd

ROOT = "/content/drive/MyDrive/RoofMaterialProject"
FEATURE_DIR = os.path.join(ROOT, "features")

# Save trained model
joblib.dump(
    best_svm_2,
    os.path.join(FEATURE_DIR, "svm_tuned_experiment2.joblib")
)

# Save cross-validation results
pd.DataFrame(grid_search_2.cv_results_).to_csv(
    os.path.join(FEATURE_DIR, "svm_experiment2_cv_results.csv"),
    index=False
)

# Save validation predictions
pd.DataFrame({
    "actual": y_val,
    "predicted": y_pred_2
}).to_csv(
    os.path.join(FEATURE_DIR, "svm_experiment2_validation_predictions.csv"),
    index=False
)

print("Experiment 2 files saved successfully!")

Experiment 2 files saved successfully!


In [20]:
import numpy as np

best_svm_1 = joblib.load(
    os.path.join(FEATURE_DIR, "svm_tuned_experiment1.joblib")
)

pred_1 = best_svm_1.predict(X_val)
pred_2 = best_svm_2.predict(X_val)

print("Identical predictions:", np.array_equal(pred_1, pred_2))
print(
    "Number of different predictions:",
    np.sum(pred_1 != pred_2)
)

Identical predictions: True
Number of different predictions: 0


In [21]:
print(
    best_svm_1.named_steps["svm"].class_weight
)

balanced


In [22]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

class_weights = {
    "balanced": "balanced",
    "no_weight": None,
    "custom": {
        "concrete_cement": 3.0,
        "healthy_metal": 0.5,
        "incomplete": 2.5,
        "irregular_metal": 1.0
    }
}

results = []

for name, weights in class_weights.items():

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            kernel="rbf",
            C=10,
            gamma="scale",
            class_weight=weights
        ))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    results.append({
        "strategy": name,
        "accuracy": accuracy_score(y_val, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_val, predictions),
        "macro_f1": f1_score(
            y_val, predictions, average="macro"
        )
    })

    print(f"\n{name.upper()}")
    print(classification_report(y_val, predictions))

results_df = pd.DataFrame(results)
print(results_df)


BALANCED
                 precision    recall  f1-score   support

concrete_cement       0.21      0.42      0.28        99
  healthy_metal       0.81      0.74      0.77      1109
     incomplete       0.31      0.47      0.37       132
irregular_metal       0.58      0.52      0.55       727

       accuracy                           0.63      2067
      macro avg       0.48      0.54      0.49      2067
   weighted avg       0.67      0.63      0.64      2067


NO_WEIGHT
                 precision    recall  f1-score   support

concrete_cement       0.43      0.15      0.22        99
  healthy_metal       0.75      0.84      0.79      1109
     incomplete       0.47      0.16      0.24       132
irregular_metal       0.60      0.61      0.60       727

       accuracy                           0.68      2067
      macro avg       0.56      0.44      0.46      2067
   weighted avg       0.66      0.68      0.66      2067


CUSTOM
                 precision    recall  f1-score   supp

In [23]:
import os
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

ROOT = "/content/drive/MyDrive/RoofMaterialProject"
FEATURE_DIR = os.path.join(ROOT, "features")

custom_weights = {
    "concrete_cement": 3.0,
    "healthy_metal": 0.5,
    "incomplete": 2.5,
    "irregular_metal": 1.0
}

svm_custom_final = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        class_weight=custom_weights
    ))
])

svm_custom_final.fit(X_train, y_train)

# Validate
y_pred_custom = svm_custom_final.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred_custom))
print("Balanced accuracy:", balanced_accuracy_score(y_val, y_pred_custom))
print("Macro-F1:", f1_score(y_val, y_pred_custom, average="macro"))

# Save
model_path = os.path.join(
    FEATURE_DIR,
    "svm_custom_4class_final.joblib"
)

joblib.dump(svm_custom_final, model_path)

print("\nSaved:", model_path)
print("Input features:", svm_custom_final.n_features_in_)
print("Classes:", svm_custom_final.classes_)

Accuracy: 0.6560232220609579
Balanced accuracy: 0.541647997645445
Macro-F1: 0.523515843483775

Saved: /content/drive/MyDrive/RoofMaterialProject/features/svm_custom_4class_final.joblib
Input features: 44
Classes: ['concrete_cement' 'healthy_metal' 'incomplete' 'irregular_metal']


In [24]:
print(feature_cols)
print("Total features:", len(feature_cols))

['r_mean', 'r_std', 'r_p10', 'r_p90', 'g_mean', 'g_std', 'g_p10', 'g_p90', 'b_mean', 'b_std', 'b_p10', 'b_p90', 'h_mean', 'h_std', 's_mean', 's_std', 'v_mean', 'v_std', 'lbp_bin_0', 'lbp_bin_1', 'lbp_bin_2', 'lbp_bin_3', 'lbp_bin_4', 'lbp_bin_5', 'lbp_bin_6', 'lbp_bin_7', 'lbp_bin_8', 'lbp_bin_9', 'glcm_contrast_d1_mean', 'glcm_contrast_d1_std', 'glcm_contrast_d2_mean', 'glcm_contrast_d2_std', 'glcm_homogeneity_d1_mean', 'glcm_homogeneity_d1_std', 'glcm_homogeneity_d2_mean', 'glcm_homogeneity_d2_std', 'glcm_energy_d1_mean', 'glcm_energy_d1_std', 'glcm_energy_d2_mean', 'glcm_energy_d2_std', 'glcm_correlation_d1_mean', 'glcm_correlation_d1_std', 'glcm_correlation_d2_mean', 'glcm_correlation_d2_std']
Total features: 44
